In [16]:
# data_merge.py
"""
Streamlined Data Merging Pipeline for Multi-Disease Prediction Project
CS-245 Course Project: Health & Public Safety Intelligence Theme
Combines BRFSS 2023 and EPA AQI 2023 datasets with advanced feature engineering
"""

import pandas as pd
import numpy as np
import os
import logging
from typing import List, Optional
import seaborn as sns
import matplotlib.pyplot as plt

# Set up logging
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')
logger = logging.getLogger(__name__)

# ==========================================
# 1. CONFIGURATION & CONSTANTS
# ==========================================

# Comprehensive BRFSS variable mapping - leveraging the 350+ columns
COMPREHENSIVE_VAR_MAP = {
    # --- GEOGRAPHY & DEMOGRAPHICS ---
    'State_FIPS': ['_STATE', 'STATE', 'FIPS'],
    'County_FIPS': ['_COUNTY', 'COUNTY', 'COUNTYFIPS', 'CNTYCOD'],
    'Age': ['_AGE80', '_AGE_G', 'AGE', '_AGEG5YR', 'AGE_G'],
    'Sex': ['SEXVAR', 'SEX', '_SEX'],  
    'Race': ['_RACE', 'RACE', '_RACEGR3', '_RACEG21', '_RACEPRV'],
    'Education': ['EDUCA', '_EDUCAG', 'EDUCAG'],   
    'Income': ['INCOME3', 'INCOME2', 'INCOME', '_INCOMG', 'INCOMG'], 
    'MaritalStatus': ['MARITAL', 'MARITAL'], 
    'Employment': ['EMPLOY1', 'EMPLOY', '_EMPLOY'],     
    'Veteran': ['VETERAN3', 'VETERAN'], 
    'Children': ['CHILDREN', 'CHLDCNT'],
    
    # --- VITAL SIGNS & ANTHROPOMETRICS ---
    'BMI': ['_BMI5', 'BMI', '_BMI5CAT'],
    'Height': ['HEIGHT3', 'HTIN4', 'HTM4'],
    'Weight': ['WEIGHT2', 'WTKG3'],
    'HighBP': ['_RFHYPE5', '_RFHYPE6', 'BPHIGH4', 'BPMEDS'],  
    'HighChol': ['_RFCHOL', '_RFCHOL1', 'TOLDHI2', 'TOLDHI3', 'CHOLCHK'], 
    'CholCheck': ['CHOLCHK', 'CHOLCHK2', 'CHOLCHK3'],
    
    # --- LIFESTYLE & BEHAVIORAL RISK FACTORS ---
    'Smoker': ['_RFSMOK3', 'SMOKER3', '_SMOKER3', '_SMKL3'], 
    'SmokeStatus': ['_SMOKER3', 'SMOKE100', 'SMOKDAY2'],
    'HeavyDrinker': ['_RFDRHV7', '_RFDRHV5', '_RFDRHV8', '_DRNKWEK'], 
    'AlcoholConsumption': ['_DRNKWEK', 'ALCDAY5', 'AVEDRNK2', 'DRNK3GE5'],
    'PhysicalActivity': ['_TOTINDA', '_PACAT1', '_PAINDX1', '_PA150R2'], 
    'FruitCons': ['_FRUTSUM', '_FRUTSU1', 'FRUITJU2', 'FRUIT2'], 
    'VegCons': ['_VEGESUM', '_VEGESU1', 'VEGETAB2', 'VEGEDA2'],   
    'SleepHours': ['SLEPTIM1', 'SLEPTIM2', '_SLEPTIM'],
    'SedentaryTime': ['_TOTINDX', '_METSTAT', 'SEDENTARY'],
    
    # --- HEALTHCARE ACCESS & UTILIZATION ---
    'AnyHealthcare': ['HLTHPLN1', 'PERSDOC2', 'MEDCOST', 'HLTHCVR'],
    'Checkup': ['CHECKUP1', 'CHECKUP', 'LASTDEN4'],
    'DentalVisit': ['LASTDEN3', 'LASTDEN4'],
    'FluShot': ['FLUSHOT7', 'FLUSHOT6'],
    'HIVTest': ['HIVTST7', 'HIVTST6'],
    
    # --- GENERAL HEALTH & FUNCTIONAL STATUS ---
    'GenHealth': ['GENHLTH', '_RFHLTH'], 
    'PhysicalHealthDays': ['PHYSHLTH', '_PHYS14D'], 
    'MentalHealthDays': ['MENTHLTH', '_MENT14D'],   
    'PoorHealthDays': ['POORHLTH', '_PHYSHLTH'],
    'Blind': ['BLIND', 'CHCVISN1', 'VISIONDF'],      
    'Deaf': ['DEAF', 'CHCHEAR1', 'HEARINGDF'],        
    'CognitiveDiff': ['DECIDE', 'DIFFCARE'],         
    'DiffWalking': ['DIFFWALK', 'DIFFALON'],         
    'DiffDressing': ['DIFFDRES'],        
    'DiffErrands': ['DIFFALON'],
    'PainArthritis': ['PAINACT2', 'ARTHDIS2'],
    
    # --- DISEASE TARGETS (PRIMARY) ---
    'HeartDisease': ['CVDCRHD4', 'MICHD', '_MICHD', 'CVDCRHD3'],  # Coronary heart disease
    'HeartAttack': ['CVDINFR4', 'CVDINFR3', 'CVDINFR'],
    'Stroke': ['CVDSTRK3', 'CVDSTRK', '_CVDSTRK'],
    'Asthma': ['ASTHMA3', 'ASTHMA', '_ASTHMS1'],
    'SkinCancer': ['CHCSCNCR', 'CHCSCNCR'],
    'OtherCancer': ['CHCOCNCR', 'CHCOCNCR'],
    'COPD': ['CHCCOPD3', 'CHCCOPD2', 'CHCCOPD', '_LTASTH1'],
    'Depression': ['ADDEPEV3', 'ADDEPEV2', '_ADDEPEV'],
    'KidneyDisease': ['CHCKDNY2', 'CHCKDNY1', '_CHCKDNY'],
    'Diabetes': ['DIABETE4', 'DIABETE3', 'DIABETE2', '_DIABETE'],
    'Arthritis': ['HAVARTH5', 'HAVARTH4', 'HAVARTH3', '_DRARTHR'],
    'Disability': ['_DISPAR', '_DIFFALON', '_DIFFWALK', '_DIFFDRS'],
    
    # --- COMORBIDITIES & RISK FACTORS ---
    'Obesity': ['_RFBMI5', '_BMI5CAT'],
    'HighRisk': ['_RFDRACU', '_RFDRHM7'],
    'Exercise': ['EXERANY2', '_EXERANY2'],
    'SaltIntake': ['SODADAY2', '_SODAIND'],
}

# FIPS to state mapping
FIPS_TO_STATE = {
    1: 'Alabama', 2: 'Alaska', 4: 'Arizona', 5: 'Arkansas', 6: 'California',
    8: 'Colorado', 9: 'Connecticut', 10: 'Delaware', 11: 'District of Columbia',
    12: 'Florida', 13: 'Georgia', 15: 'Hawaii', 16: 'Idaho', 17: 'Illinois',
    18: 'Indiana', 19: 'Iowa', 20: 'Kansas', 21: 'Kentucky', 22: 'Louisiana',
    23: 'Maine', 24: 'Maryland', 25: 'Massachusetts', 26: 'Michigan',
    27: 'Minnesota', 28: 'Mississippi', 29: 'Missouri', 30: 'Montana',
    31: 'Nebraska', 32: 'Nevada', 33: 'New Hampshire', 34: 'New Jersey',
    35: 'New Mexico', 36: 'New York', 37: 'North Carolina', 38: 'North Dakota',
    39: 'Ohio', 40: 'Oklahoma', 41: 'Oregon', 42: 'Pennsylvania',
    44: 'Rhode Island', 45: 'South Carolina', 46: 'South Dakota', 47: 'Tennessee',
    48: 'Texas', 49: 'Utah', 50: 'Vermont', 51: 'Virginia', 53: 'Washington',
    54: 'West Virginia', 55: 'Wisconsin', 56: 'Wyoming'
}

# ==========================================
# 2. CORE DATA PROCESSING FUNCTIONS
# ==========================================

def get_column_name(df: pd.DataFrame, possible_names: List[str]) -> Optional[str]:
    """Robust column finder with case-insensitive matching."""
    df_cols_upper = [str(c).upper() for c in df.columns]
    for name in possible_names:
        if name.upper() in df_cols_upper:
            return df.columns[df_cols_upper.index(name.upper())]
    return None

def create_fips_code(state_fips: pd.Series, county_fips: pd.Series) -> pd.Series:
    """Create 5-digit FIPS code from state and county codes."""
    return (state_fips.astype(str).str.zfill(2) + 
            county_fips.astype(str).str.zfill(3)).astype(str)

def clean_brfss_data(file_path: str) -> pd.DataFrame:
    """
    Load and clean BRFSS data with advanced feature extraction.
    
    Args:
        file_path: Path to BRFSS CSV file
        
    Returns:
        Cleaned individual-level DataFrame with engineered features
    """
    logger.info(f"Loading and processing BRFSS data from {file_path}")
    
    try:
        # Load BRFSS data - read in chunks if needed
        df = pd.read_csv(file_path, low_memory=False, encoding='cp1252')
        logger.info(f"Loaded {len(df):,} rows, {df.shape[1]} columns")
    except Exception as e:
        logger.error(f"Failed to load BRFSS data: {e}")
        return pd.DataFrame()
    
    # Initialize clean DataFrame
    clean_data = pd.DataFrame()
    
    # 1. Extract columns using comprehensive variable mapping
    logger.info("Extracting variables from BRFSS...")
    extracted_count = 0
    for target_name, possible_cols in COMPREHENSIVE_VAR_MAP.items():
        raw_col = get_column_name(df, possible_cols)
        if raw_col and raw_col in df.columns:
            clean_data[target_name] = df[raw_col]
            extracted_count += 1
        else:
            logger.debug(f"Column not found for {target_name}: {possible_cols}")
            clean_data[target_name] = np.nan
    
    logger.info(f"Extracted {extracted_count} out of {len(COMPREHENSIVE_VAR_MAP)} variables")
    
    # 2. Create composite FIPS code
    if 'State_FIPS' in clean_data.columns and 'County_FIPS' in clean_data.columns:
        clean_data['FIPS'] = create_fips_code(clean_data['State_FIPS'], 
                                             clean_data['County_FIPS'])
        clean_data['State'] = clean_data['State_FIPS'].map(FIPS_TO_STATE)
    else:
        logger.error("Missing State_FIPS or County_FIPS columns")
        return pd.DataFrame()
    
    # 3. Clean and transform variables
    logger.info("Cleaning and transforming variables...")
    
    # Disease targets: binary encoding
    disease_targets = [
        'HeartDisease', 'HeartAttack', 'Stroke', 'Asthma', 'SkinCancer', 
        'OtherCancer', 'COPD', 'Depression', 'KidneyDisease', 'Diabetes', 
        'Arthritis'
    ]
    
    for col in [c for c in disease_targets if c in clean_data.columns]:
        # Convert to numeric, coerce errors
        clean_data[col] = pd.to_numeric(clean_data[col], errors='coerce')
        # BRFSS coding: 1=Yes, 2=No, 7/9=Don't know/Refused
        clean_data[col] = clean_data[col].apply(
            lambda x: 1 if x == 1 else (0 if pd.notna(x) else np.nan)
        )
    
    # Risk factors: binary encoding
    risk_factors = [
        'HighBP', 'HighChol', 'Smoker', 'HeavyDrinker', 'PhysicalActivity',
        'AnyHealthcare', 'CholCheck', 'Obesity', 'HighRisk', 'Exercise'
    ]
    
    for col in [c for c in risk_factors if c in clean_data.columns]:
        clean_data[col] = pd.to_numeric(clean_data[col], errors='coerce')
        clean_data[col] = clean_data[col].apply(
            lambda x: 1 if x == 1 else (0 if pd.notna(x) else np.nan)
        )
    
    # Continuous variables
    if 'BMI' in clean_data.columns:
        clean_data['BMI'] = pd.to_numeric(clean_data['BMI'], errors='coerce') / 100
        # Create BMI categories
        clean_data['BMI_Category'] = pd.cut(
            clean_data['BMI'],
            bins=[0, 18.5, 25, 30, 35, 40, 100],
            labels=['Underweight', 'Normal', 'Overweight', 'Obese I', 'Obese II', 'Obese III']
        )
    
    if 'Age' in clean_data.columns:
        clean_data['Age'] = pd.to_numeric(clean_data['Age'], errors='coerce')
        clean_data['Age_Group'] = pd.cut(
            clean_data['Age'],
            bins=[0, 18, 25, 35, 45, 55, 65, 75, 85, 100],
            labels=['<18', '18-24', '25-34', '35-44', '45-54', '55-64', '65-74', '75-84', '85+']
        )
    
    # Health behavior scores
    if all(c in clean_data.columns for c in ['FruitCons', 'VegCons']):
        clean_data['HealthyDiet_Score'] = (
            pd.to_numeric(clean_data['FruitCons'], errors='coerce').fillna(0) +
            pd.to_numeric(clean_data['VegCons'], errors='coerce').fillna(0)
        ) / 2
    
    if all(c in clean_data.columns for c in ['MentalHealthDays', 'PhysicalHealthDays']):
        clean_data['TotalUnhealthyDays'] = (
            pd.to_numeric(clean_data['MentalHealthDays'], errors='coerce').fillna(0) +
            pd.to_numeric(clean_data['PhysicalHealthDays'], errors='coerce').fillna(0)
        ).clip(upper=30)
    
    # 4. Create advanced composite features
    logger.info("Creating advanced composite features...")
    
    # Multi-morbidity index
    disease_cols_for_index = [c for c in disease_targets if c in clean_data.columns]
    if disease_cols_for_index:
        clean_data['MultiMorbidity_Index'] = clean_data[disease_cols_for_index].sum(axis=1)
        clean_data['HasMultipleConditions'] = (clean_data['MultiMorbidity_Index'] >= 2).astype(int)
    
    # Cardiovascular risk score (simplified)
    cv_risk_factors = []
    if 'HighBP' in clean_data.columns:
        cv_risk_factors.append('HighBP')
    if 'HighChol' in clean_data.columns:
        cv_risk_factors.append('HighChol')
    if 'Smoker' in clean_data.columns:
        cv_risk_factors.append('Smoker')
    if 'Diabetes' in clean_data.columns:
        cv_risk_factors.append('Diabetes')
    
    if cv_risk_factors:
        clean_data['CardioRisk_Score'] = clean_data[cv_risk_factors].sum(axis=1)
    
    # Lifestyle risk score
    lifestyle_factors = []
    if 'Smoker' in clean_data.columns:
        lifestyle_factors.append('Smoker')
    if 'HeavyDrinker' in clean_data.columns:
        lifestyle_factors.append('HeavyDrinker')
    if 'PhysicalActivity' in clean_data.columns:
        # Invert physical activity (1=active, so 0=risk)
        clean_data['PhysicalInactivity'] = 1 - clean_data['PhysicalActivity']
        lifestyle_factors.append('PhysicalInactivity')
    
    if lifestyle_factors:
        clean_data['LifestyleRisk_Score'] = clean_data[lifestyle_factors].sum(axis=1)
    
    # Healthcare access composite
    if 'AnyHealthcare' in clean_data.columns:
        clean_data['HealthcareAccess_Score'] = clean_data['AnyHealthcare']
        if 'Checkup' in clean_data.columns:
            # Checkup in last year (1=Yes) is good
            clean_data['HealthcareAccess_Score'] += clean_data['Checkup'].apply(
                lambda x: 1 if x == 1 else 0
            )
    
    logger.info(f"BRFSS processing complete: {len(clean_data):,} individuals")
    return clean_data

def aggregate_to_county_level(individual_data: pd.DataFrame, min_sample_size: int = 30) -> pd.DataFrame:
    """
    Aggregate individual-level data to county level with comprehensive statistics.
    
    Args:
        individual_data: Cleaned individual-level DataFrame
        min_sample_size: Minimum number of respondents per county
        
    Returns:
        County-level aggregated DataFrame
    """
    logger.info("Aggregating data to county level...")
    
    if individual_data.empty:
        return pd.DataFrame()
    
    # Group by FIPS
    county_groups = individual_data.groupby('FIPS')
    
    # Calculate sample size per county
    county_sample_sizes = county_groups.size().reset_index(name='Sample_Size')
    
    # Filter counties with sufficient sample size
    valid_counties = county_sample_sizes[county_sample_sizes['Sample_Size'] >= min_sample_size]['FIPS']
    filtered_data = individual_data[individual_data['FIPS'].isin(valid_counties)]
    
    logger.info(f"Retaining {len(valid_counties)} counties with ≥{min_sample_size} respondents")
    
    # Define aggregation strategies
    county_stats = []
    
    for fips in valid_counties:
        county_df = filtered_data[filtered_data['FIPS'] == fips]
        county_info = {'FIPS': fips, 'Sample_Size': len(county_df)}
        
        # Get state from first record
        if 'State' in county_df.columns:
            county_info['State'] = county_df['State'].iloc[0]
        
        # Disease prevalence rates (proportions)
        disease_targets = [
            'HeartDisease', 'HeartAttack', 'Stroke', 'Asthma', 'Diabetes',
            'COPD', 'Depression', 'KidneyDisease', 'Arthritis'
        ]
        
        for disease in [d for d in disease_targets if d in county_df.columns]:
            county_info[f'Pct_{disease}'] = county_df[disease].mean() * 100
        
        # Risk factor prevalence
        risk_factors = [
            'HighBP', 'HighChol', 'Smoker', 'HeavyDrinker', 'PhysicalActivity',
            'Obesity', 'AnyHealthcare'
        ]
        
        for risk in [r for r in risk_factors if r in county_df.columns]:
            county_info[f'Pct_{risk}'] = county_df[risk].mean() * 100
        
        # Continuous variables (means)
        continuous_vars = ['BMI', 'Age', 'TotalUnhealthyDays', 'SleepHours']
        for var in [v for v in continuous_vars if v in county_df.columns]:
            county_info[f'Avg_{var}'] = county_df[var].mean()
        
        # Composite scores (means)
        composite_scores = [
            'MultiMorbidity_Index', 'CardioRisk_Score', 'LifestyleRisk_Score',
            'HealthcareAccess_Score', 'HealthyDiet_Score'
        ]
        
        for score in [s for s in composite_scores if s in county_df.columns]:
            county_info[f'Avg_{score}'] = county_df[score].mean()
        
        # Calculate variance for key metrics
        if 'BMI' in county_df.columns:
            county_info['BMI_Variance'] = county_df['BMI'].var()
        
        # Calculate correlations within county (simplified)
        if all(c in county_df.columns for c in ['Diabetes', 'HighBP']):
            county_info['Diabetes_HighBP_Correlation'] = county_df[['Diabetes', 'HighBP']].corr().iloc[0,1]
        
        county_stats.append(county_info)
    
    county_data = pd.DataFrame(county_stats)
    logger.info(f"Created county-level dataset with {len(county_data)} counties and {len(county_data.columns)} features")
    
    return county_data

def process_epa_aqi_data(file_path: str) -> pd.DataFrame:
    """
    Process EPA AQI data and create comprehensive air quality metrics.
    
    Args:
        file_path: Path to EPA AQI CSV
        
    Returns:
        Processed EPA data with enhanced AQI metrics
    """
    logger.info(f"Processing EPA AQI data from {file_path}")
    
    try:
        df = pd.read_csv(file_path)
        
        # Clean column names
        df.columns = [col.strip().replace('"', '') for col in df.columns]
        
        # Clean state and county names
        df['State'] = df['State'].str.strip()
        df['County'] = df['County'].str.replace(' County', '').str.replace(' Parish', '').str.strip()
        
        # Create enhanced AQI metrics
        df['AQI_Good_Days_Pct'] = (df['Good Days'] / df['Days with AQI']) * 100
        df['AQI_Moderate_Days_Pct'] = (df['Moderate Days'] / df['Days with AQI']) * 100
        df['AQI_Unhealthy_Days_Pct'] = (
            (df['Unhealthy for Sensitive Groups Days'] + 
             df['Unhealthy Days'] + 
             df['Very Unhealthy Days'] + 
             df['Hazardous Days']) / df['Days with AQI']
        ) * 100
        
        # AQI severity index (weighted average)
        df['AQI_Severity_Index'] = (
            (df['Good Days'] * 1 +
             df['Moderate Days'] * 2 +
             df['Unhealthy for Sensitive Groups Days'] * 3 +
             df['Unhealthy Days'] * 4 +
             df['Very Unhealthy Days'] * 5 +
             df['Hazardous Days'] * 6) / df['Days with AQI']
        )
        
        # Primary pollutant days
        pollutant_cols = ['Days CO', 'Days NO2', 'Days Ozone', 'Days PM2.5', 'Days PM10']
        available_pollutants = [col for col in pollutant_cols if col in df.columns]
        
        for pollutant in available_pollutants:
            df[f'{pollutant}_Pct'] = (df[pollutant] / df['Days with AQI']) * 100
        
        # Determine dominant pollutant
        if available_pollutants:
            pollutant_values = df[available_pollutants].values
            dominant_idx = np.argmax(pollutant_values, axis=1)
            df['Dominant_Pollutant'] = [available_pollutants[i].replace('Days ', '') for i in dominant_idx]
        
        logger.info(f"Processed EPA data for {len(df)} counties")
        
        # Select final columns
        essential_cols = ['State', 'County', 'Year', 'Max AQI', '90th Percentile AQI', 
                         'Median AQI', 'AQI_Good_Days_Pct', 'AQI_Unhealthy_Days_Pct',
                         'AQI_Severity_Index']
        
        # Add available pollutant columns
        for pollutant in ['Days Ozone_Pct', 'Days PM2.5_Pct', 'Days PM10_Pct']:
            if pollutant in df.columns:
                essential_cols.append(pollutant)
        
        return df[essential_cols]
        
    except Exception as e:
        logger.error(f"Failed to process EPA data: {e}")
        return pd.DataFrame()

def merge_brfss_epa_data(brfss_county: pd.DataFrame, epa_data: pd.DataFrame) -> pd.DataFrame:
    """
    Merge BRFSS county data with EPA AQI data.
    
    Args:
        brfss_county: BRFSS data aggregated to county level
        epa_data: Processed EPA AQI data
        
    Returns:
        Merged dataset with comprehensive features
    """
    logger.info("Merging BRFSS and EPA datasets...")
    
    # Create merging keys
    # BRFSS has FIPS codes, EPA has State and County names
    # We need to match them
    
    # For now, we'll merge on State and County names
    # In a production system, you would use FIPS codes throughout
    
    if 'State' not in brfss_county.columns or 'State' not in epa_data.columns:
        logger.error("State column missing in one of the datasets")
        return pd.DataFrame()
    
    # Create county name in BRFSS (we'll use FIPS as county identifier for now)
    # For merging, we'll need to map FIPS to county names
    
    # Simplified merge on state (demonstration)
    merged_df = brfss_county.copy()
    
    # Calculate state-level AQI averages from EPA data
    state_aqi_stats = epa_data.groupby('State').agg({
        'Median AQI': 'mean',
        'Max AQI': 'mean',
        'AQI_Unhealthy_Days_Pct': 'mean',
        'AQI_Severity_Index': 'mean'
    }).reset_index()
    
    state_aqi_stats.columns = ['State', 'State_Avg_AQI', 'State_Max_AQI', 
                               'State_Unhealthy_Days_Pct', 'State_AQI_Severity']
    
    # Merge state-level AQI stats with BRFSS data
    merged_df = merged_df.merge(state_aqi_stats, on='State', how='left')
    
    # Create interaction features between health and environment
    logger.info("Creating health-environment interaction features...")
    
    # Air quality impact on respiratory conditions
    if all(c in merged_df.columns for c in ['Pct_Asthma', 'State_Avg_AQI']):
        merged_df['Asthma_AQI_Interaction'] = merged_df['Pct_Asthma'] * merged_df['State_Avg_AQI'] / 100
    
    if all(c in merged_df.columns for c in ['Pct_COPD', 'State_Avg_AQI']):
        merged_df['COPD_AQI_Interaction'] = merged_df['Pct_COPD'] * merged_df['State_Avg_AQI'] / 100
    
    # Combined risk scores
    health_risk_cols = [c for c in merged_df.columns if c.startswith('Pct_') and 
                       any(d in c for d in ['Heart', 'Stroke', 'Diabetes'])]
    
    if health_risk_cols and 'State_Avg_AQI' in merged_df.columns:
        merged_df['Cardio_Environmental_Risk'] = (
            merged_df[health_risk_cols].mean(axis=1) * merged_df['State_Avg_AQI'] / 100
        )
    
    # Socio-environmental vulnerability index
    vulnerability_factors = []
    if 'Pct_Diabetes' in merged_df.columns:
        vulnerability_factors.append('Pct_Diabetes')
    if 'Pct_HighBP' in merged_df.columns:
        vulnerability_factors.append('Pct_HighBP')
    if 'State_Unhealthy_Days_Pct' in merged_df.columns:
        vulnerability_factors.append('State_Unhealthy_Days_Pct')
    
    if vulnerability_factors:
        # Normalize each factor
        for factor in vulnerability_factors:
            if factor in merged_df.columns:
                merged_df[f'{factor}_norm'] = (
                    merged_df[factor] - merged_df[factor].min()
                ) / (merged_df[factor].max() - merged_df[factor].min() + 1e-10)
        
        norm_cols = [f'{f}_norm' for f in vulnerability_factors if f'{f}_norm' in merged_df.columns]
        if norm_cols:
            merged_df['SocioEnvironmental_Vulnerability'] = merged_df[norm_cols].mean(axis=1)
    
    logger.info(f"Merged dataset shape: {merged_df.shape}")
    return merged_df

def perform_comprehensive_eda(df: pd.DataFrame, output_dir: str = 'reports/eda'):
    """
    Perform comprehensive exploratory data analysis with visualizations.
    
    Args:
        df: Merged DataFrame
        output_dir: Directory to save EDA reports
    """
    logger.info("Performing comprehensive EDA...")
    
    os.makedirs(output_dir, exist_ok=True)
    
    # 1. Dataset Overview
    overview_report = f"""
    ============================================
    COMPREHENSIVE EDA REPORT - MULTI-DISEASE PREDICTION
    ============================================
    
    Dataset Overview:
    - Total counties: {len(df)}
    - Total features: {len(df.columns)}
    - Missing values: {df.isnull().sum().sum():,} ({df.isnull().sum().sum()/(len(df)*len(df.columns))*100:.1f}%)
    
    Feature Categories:
    """
    
    # Categorize features
    feature_categories = {
        'Disease Prevalence': [c for c in df.columns if c.startswith('Pct_') and 
                              any(d in c for d in ['Heart', 'Stroke', 'Asthma', 'Diabetes', 'COPD', 'Depression'])],
        'Risk Factors': [c for c in df.columns if c.startswith('Pct_') and 
                        any(r in c for r in ['Smoker', 'HighBP', 'HighChol', 'Obesity'])],
        'Demographics': [c for c in df.columns if 'Age' in c or 'Sample_Size' in c],
        'Environmental': [c for c in df.columns if 'AQI' in c or 'Pollutant' in c],
        'Composite Scores': [c for c in df.columns if 'Score' in c or 'Index' in c or 'Interaction' in c]
    }
    
    for category, features in feature_categories.items():
        if features:
            overview_report += f"\n  {category}: {len(features)} features"
            # Example features
            overview_report += f"\n    Examples: {', '.join(features[:3])}"
    
    # 2. Statistical Summary
    numeric_cols = df.select_dtypes(include=[np.number]).columns
    stats_summary = df[numeric_cols].describe().T
    stats_summary.to_csv(f'{output_dir}/statistical_summary.csv')
    
    # 3. Correlation Analysis
    if len(numeric_cols) > 0:
        # Calculate correlation matrix
        corr_matrix = df[numeric_cols].corr()
        corr_matrix.to_csv(f'{output_dir}/correlation_matrix.csv')
        
        # Top correlations with key diseases
        key_diseases = ['Pct_Diabetes', 'Pct_HeartDisease', 'Pct_Stroke', 'Pct_Asthma']
        existing_diseases = [d for d in key_diseases if d in corr_matrix.columns]
        
        if existing_diseases:
            disease_correlations = pd.DataFrame()
            for disease in existing_diseases:
                # Get top 10 positive and negative correlations
                corr_with_disease = corr_matrix[disease].drop(disease)
                top_positive = corr_with_disease.nlargest(10)
                top_negative = corr_with_disease.nsmallest(10)
                
                disease_correlations[f'{disease}_Positive'] = top_positive
                disease_correlations[f'{disease}_Negative'] = top_negative
            
            disease_correlations.to_csv(f'{output_dir}/disease_correlations.csv')
    
    # 4. Missing Values Analysis
    missing_data = pd.DataFrame({
        'Feature': df.columns,
        'Missing_Count': df.isnull().sum(),
        'Missing_Percentage': (df.isnull().sum() / len(df)) * 100,
        'Data_Type': df.dtypes
    }).sort_values('Missing_Percentage', ascending=False)
    
    missing_data.to_csv(f'{output_dir}/missing_values_analysis.csv', index=False)
    
    # 5. Distribution Plots (save as images)
    plt.figure(figsize=(15, 10))
    
    # Plot distributions of key disease prevalence
    disease_cols = [c for c in df.columns if c.startswith('Pct_') and 
                   any(d in c for d in ['Diabetes', 'HeartDisease', 'Stroke', 'Asthma'])]
    
    if disease_cols:
        fig, axes = plt.subplots(2, 2, figsize=(12, 10))
        axes = axes.flatten()
        
        for idx, disease in enumerate(disease_cols[:4]):
            if disease in df.columns:
                axes[idx].hist(df[disease].dropna(), bins=30, alpha=0.7, color='skyblue', edgecolor='black')
                axes[idx].set_title(f'Distribution of {disease}')
                axes[idx].set_xlabel('Prevalence (%)')
                axes[idx].set_ylabel('Frequency')
        
        plt.tight_layout()
        plt.savefig(f'{output_dir}/disease_distributions.png', dpi=150, bbox_inches='tight')
        plt.close()
    
    # 6. Save comprehensive report
    with open(f'{output_dir}/comprehensive_eda_report.txt', 'w') as f:
        f.write(overview_report)
        f.write("\n\n" + "="*60 + "\n")
        f.write("DATA QUALITY ASSESSMENT\n")
        f.write("="*60 + "\n")
        f.write(f"\nFeatures with >20% missing values: {len(missing_data[missing_data['Missing_Percentage'] > 20])}")
        f.write(f"\nFeatures with >50% missing values: {len(missing_data[missing_data['Missing_Percentage'] > 50])}")
        
        f.write("\n\n" + "="*60 + "\n")
        f.write("KEY INSIGHTS\n")
        f.write("="*60 + "\n")
        
        # Calculate basic insights
        if 'State_Avg_AQI' in df.columns:
            avg_aqi = df['State_Avg_AQI'].mean()
            f.write(f"\nAverage State AQI: {avg_aqi:.1f}")
        
        if 'Pct_Diabetes' in df.columns:
            avg_diabetes = df['Pct_Diabetes'].mean()
            f.write(f"\nAverage Diabetes Prevalence: {avg_diabetes:.1f}%")
        
        # Top correlations if available
        if 'disease_correlations' in locals():
            f.write("\n\nTop Disease Correlations:")
            for disease in existing_diseases[:2]:
                f.write(f"\n\n{disease}:")
                if f'{disease}_Positive' in disease_correlations.columns:
                    top_corr = disease_correlations[f'{disease}_Positive'].head(3)
                    for feature, value in top_corr.items():
                        f.write(f"\n  + {feature}: {value:.3f}")
    
    logger.info(f"EDA reports saved to {output_dir}")

# ==========================================
# 3. MAIN EXECUTION PIPELINE
# ==========================================

def main():
    """Main function to execute the complete data processing pipeline."""
    logger.info("=" * 70)
    logger.info("CS-245 PROJECT: ADVANCED MULTI-DISEASE PREDICTION PIPELINE")
    logger.info("=" * 70)
    
    # Create directories
    os.makedirs('data/raw', exist_ok=True)
    os.makedirs('data/processed', exist_ok=True)
    os.makedirs('reports/eda', exist_ok=True)
    os.makedirs('models', exist_ok=True)
    
    # 1. Process BRFSS data
    logger.info("\n" + "="*70)
    logger.info("PHASE 1: PROCESSING BRFSS 2023 DATA")
    logger.info("="*70)
    
    brfss_file = 'data/raw/BRFSS2023.csv'
    if not os.path.exists(brfss_file):
        logger.error(f"BRFSS file not found: {brfss_file}")
        logger.info("Please ensure BRFSS2023.csv is in data/raw/ directory")
        return
    
    # Load and clean individual-level BRFSS data
    brfss_individual = clean_brfss_data(brfss_file)
    if brfss_individual.empty:
        logger.error("Failed to process BRFSS data")
        return
    
    # Save individual data for reference
    brfss_individual.to_csv('data/processed/brfss_individual_cleaned.csv', index=False)
    logger.info(f"Saved individual-level data: {len(brfss_individual):,} records")
    
    # 2. Aggregate to county level
    logger.info("\n" + "="*70)
    logger.info("PHASE 2: AGGREGATING TO COUNTY LEVEL")
    logger.info("="*70)
    
    brfss_county = aggregate_to_county_level(brfss_individual, min_sample_size=30)
    if brfss_county.empty:
        logger.error("Failed to aggregate data to county level")
        return
    
    # Save county-level BRFSS data
    brfss_county.to_csv('data/processed/brfss_county_aggregated.csv', index=False)
    logger.info(f"Saved county-level BRFSS data: {len(brfss_county)} counties")
    
    # 3. Process EPA AQI data
    logger.info("\n" + "="*70)
    logger.info("PHASE 3: PROCESSING EPA AQI DATA")
    logger.info("="*70)
    
    epa_file = 'data/raw/annual_aqi_by_county_2023.csv'
    epa_data = process_epa_aqi_data(epa_file)
    
    if epa_data.empty:
        logger.warning("EPA data not available, using BRFSS data only")
        merged_data = brfss_county
    else:
        # 4. Merge datasets
        logger.info("\n" + "="*70)
        logger.info("PHASE 4: MERGING BRFSS AND EPA DATA")
        logger.info("="*70)
        
        merged_data = merge_brfss_epa_data(brfss_county, epa_data)
    
    # 5. Perform comprehensive EDA
    logger.info("\n" + "="*70)
    logger.info("PHASE 5: EXPLORATORY DATA ANALYSIS")
    logger.info("="*70)
    
    perform_comprehensive_eda(merged_data)
    
    # 6. Save final dataset
    logger.info("\n" + "="*70)
    logger.info("PHASE 6: SAVING FINAL DATASET")
    logger.info("="*70)
    
    output_file = 'data/processed/multi_disease_final_dataset_2023.csv'
    merged_data.to_csv(output_file, index=False)
    
    # 7. Generate comprehensive summary
    logger.info("\n" + "="*70)
    logger.info("PIPELINE COMPLETE - DATASET SUMMARY")
    logger.info("="*70)
    
    logger.info(f"Final dataset shape: {merged_data.shape}")
    logger.info(f"Number of counties: {len(merged_data)}")
    logger.info(f"Number of features: {len(merged_data.columns)}")
    
    # Feature category breakdown
    feature_categories = {
        'Disease Prevalence': len([c for c in merged_data.columns if c.startswith('Pct_') and 
                                  any(d in c for d in ['Heart', 'Stroke', 'Asthma', 'Diabetes', 'COPD', 'Depression'])]),
        'Risk Factors': len([c for c in merged_data.columns if c.startswith('Pct_') and 
                            any(r in c for r in ['Smoker', 'HighBP', 'HighChol', 'Obesity'])]),
        'Environmental': len([c for c in merged_data.columns if 'AQI' in c]),
        'Composite Scores': len([c for c in merged_data.columns if 'Score' in c or 'Index' in c]),
        'Demographics': len([c for c in merged_data.columns if 'Age' in c or 'Sample' in c or 'State' in c])
    }
    
    logger.info("\nFeature Categories:")
    for category, count in feature_categories.items():
        if count > 0:
            logger.info(f"  {category}: {count} features")
    
    # Disease prevalence summary
    logger.info("\nAverage Disease Prevalence by County:")
    disease_cols = [c for c in merged_data.columns if c.startswith('Pct_') and 
                   any(d in c for d in ['Diabetes', 'HeartDisease', 'Stroke', 'Asthma', 'COPD', 'Depression'])]
    
    for disease in disease_cols:
        if disease in merged_data.columns:
            mean_val = merged_data[disease].mean()
            std_val = merged_data[disease].std()
            logger.info(f"  {disease}: {mean_val:.1f}% (±{std_val:.1f}%)")
    
    # Data quality metrics
    missing_pct = (merged_data.isnull().sum().sum() / (len(merged_data) * len(merged_data.columns))) * 100
    logger.info(f"\nData Quality:")
    logger.info(f"  Missing values: {missing_pct:.2f}%")
    logger.info(f"  Complete cases: {merged_data.dropna().shape[0]} counties ({merged_data.dropna().shape[0]/len(merged_data)*100:.1f}%)")
    
    logger.info(f"\nOutput files:")
    logger.info(f"  Final dataset: {output_file}")
    logger.info(f"  Individual BRFSS: data/processed/brfss_individual_cleaned.csv")
    logger.info(f"  County BRFSS: data/processed/brfss_county_aggregated.csv")
    logger.info(f"  EDA reports: reports/eda/")
    
    # Create README for the dataset
    create_dataset_readme(merged_data)

def create_dataset_readme(df: pd.DataFrame):
    """Create a detailed README for the processed dataset."""
    readme_file = 'data/processed/README.md'
    
    with open(readme_file, 'w') as f:
        f.write("# Multi-Disease Prediction Dataset 2023\n\n")
        f.write("## Overview\n")
        f.write("This dataset combines BRFSS 2023 health survey data with EPA 2023 air quality data ")
        f.write("for county-level multi-disease risk prediction in the United States.\n\n")
        
        f.write("## Data Sources\n")
        f.write("1. **BRFSS 2023**: Behavioral Risk Factor Surveillance System (CDC)\n")
        f.write("   - 400,000+ individual responses\n")
        f.write("   - 350+ variables covering demographics, behaviors, and health outcomes\n")
        f.write("   - Aggregated to county level with ≥30 respondents per county\n\n")
        
        f.write("2. **EPA AQI 2023**: Air Quality Index Data (EPA)\n")
        f.write("   - Daily AQI measurements aggregated to annual county-level statistics\n")
        f.write("   - Includes Good/Moderate/Unhealthy days, max AQI, median AQI\n")
        f.write("   - Pollutant-specific days (Ozone, PM2.5, PM10)\n\n")
        
        f.write("## Dataset Statistics\n")
        f.write(f"- Counties: {len(df)}\n")
        f.write(f"- Features: {len(df.columns)}\n")
        f.write(f"- Time period: 2023\n")
        f.write(f"- Geographic coverage: Contiguous United States\n")
        f.write(f"- Missing values: {(df.isnull().sum().sum()/(len(df)*len(df.columns))*100):.1f}%\n\n")
        
        f.write("## Feature Engineering\n")
        f.write("### Created Features:\n")
        f.write("1. **Disease Prevalence Rates**: Percentage of population with each disease\n")
        f.write("2. **Risk Factor Prevalence**: Smoking, hypertension, high cholesterol rates\n")
        f.write("3. **Composite Health Scores**: Multi-morbidity, cardiovascular risk, lifestyle risk\n")
        f.write("4. **Environmental Metrics**: AQI severity, unhealthy days percentage\n")
        f.write("5. **Interaction Features**: Disease-AQI interactions, socio-environmental vulnerability\n\n")
        
        f.write("## Key Variables\n")
        f.write("| Category | Key Variables | Description |\n")
        f.write("|----------|---------------|-------------|\n")
        f.write("| **Targets** | Pct_Diabetes, Pct_HeartDisease, Pct_Stroke, Pct_Asthma | Disease prevalence rates |\n")
        f.write("| **Risk Factors** | Pct_Smoker, Pct_HighBP, Pct_HighChol, Pct_Obesity | Behavioral and clinical risk factors |\n")
        f.write("| **Environmental** | State_Avg_AQI, State_Unhealthy_Days_Pct, AQI_Severity_Index | Air quality indicators |\n")
        f.write("| **Composite** | MultiMorbidity_Index, CardioRisk_Score, LifestyleRisk_Score | Derived health risk scores |\n")
        f.write("| **Interaction** | Asthma_AQI_Interaction, SocioEnvironmental_Vulnerability | Health-environment interactions |\n\n")
        
        f.write("## Usage\n")
        f.write("This dataset is designed for:\n")
        f.write("1. Multi-disease risk prediction models\n")
        f.write("2. Health disparity analysis by county\n")
        f.write("3. Environmental health impact studies\n")
        f.write("4. Public health intervention planning\n\n")
        
        f.write("## File Structure\n")
        f.write("```\n")
        f.write("project/\n")
        f.write("├── data/raw/                    # Original data files\n")
        f.write("│   ├── BRFSS2023.csv           # BRFSS 2023 data (400K+ rows)\n")
        f.write("│   └── annual_aqi_by_county_2023.csv  # EPA AQI 2023 data\n")
        f.write("├── data/processed/             # Processed datasets\n")
        f.write("│   ├── brfss_individual_cleaned.csv    # Cleaned individual data\n")
        f.write("│   ├── brfss_county_aggregated.csv     # County-aggregated BRFSS\n")
        f.write("│   ├── multi_disease_final_dataset_2023.csv  # Final merged dataset\n")
        f.write("│   └── README.md               # This file\n")
        f.write("└── reports/eda/                # Exploratory Data Analysis reports\n")
        f.write("    ├── statistical_summary.csv\n")
        f.write("    ├── correlation_matrix.csv\n")
        f.write("    ├── disease_correlations.csv\n")
        f.write("    ├── missing_values_analysis.csv\n")
        f.write("    └── comprehensive_eda_report.txt\n")
        f.write("```\n")

if __name__ == "__main__":
    main()

2025-12-14 15:10:50,183 - INFO - ======================================================================
2025-12-14 15:10:50,184 - INFO - CS-245 PROJECT: ADVANCED MULTI-DISEASE PREDICTION PIPELINE
2025-12-14 15:10:50,184 - INFO - ======================================================================
2025-12-14 15:10:50,185 - INFO - 
2025-12-14 15:10:50,186 - INFO - PHASE 1: PROCESSING BRFSS 2023 DATA
2025-12-14 15:10:50,186 - INFO - ======================================================================
2025-12-14 15:10:50,186 - INFO - Loading and processing BRFSS data from data/raw/BRFSS2023.csv
2025-12-14 15:11:06,279 - INFO - Loaded 433,323 rows, 350 columns
2025-12-14 15:11:06,283 - INFO - Extracting variables from BRFSS...
2025-12-14 15:11:06,374 - INFO - Extracted 47 out of 58 variables
2025-12-14 15:11:06,668 - INFO - Cleaning and transforming variables...
2025-12-14 15:11:09,980 - INFO - Creating advanced composite features...
2025-12-14 15:11:10,270 - INFO - BRFSS processing com

<Figure size 1500x1000 with 0 Axes>